<a href="https://colab.research.google.com/github/nalinegwizaigihozo/Clinic-KnowledgeResearch/blob/main/person1_agent_orchestrator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Person 1: Agentic Architecture & Orchestration Lead
**One notebook, just click Run All.** Works in Google Colab or Jupyter, no folders to upload.

What this notebook covers (your 4 responsibilities):
1. **Initialize the LangGraph ReAct agent** (section 4)
2. **System prompt and behavioural constraints**: research tool, no diagnoses (section 2)
3. **Orchestrator loop**: the agent decides between the graph tool and the vector tool (sections 3 and 4)
4. **Test multi-step reasoning prompts** (sections 5 and 6)

**Two modes, chosen automatically**
- **No API key:** an *offline stand-in model* (simple keyword rules) plays the role of the LLM. The whole LangGraph agent, prompt, tools and loop still run, so everything works with no key.
- **With a free Gemini key:** the real LLM does the reasoning. In Colab, add the key under the key icon (Secrets) with the name `GOOGLE_API_KEY`. Elsewhere, set the environment variable `GOOGLE_API_KEY`. Then run again.

## Team contract (all 5 notebooks follow this)
| Piece | Owner | Signature | Returns |
|---|---|---|---|
| `_graph(question)` | Person 2 | `str -> str` | `GRAPH RESULT:` + `(A)-[:REL]->(B)` lines + `SOURCE: graph:...`, or `NO RESULTS...` |
| `_vector(query)` | Person 3 | `str -> str` | `LITERATURE RESULT n:` + text on one line + `SOURCE: PMID:...`, or `NO RESULTS...` |
| `run_agent(text, thread_id)` | Person 1 (this notebook) | `-> dict` | `answer, sources, tool_calls, graph_nodes, graph_edges, trace` |
| config, error handling, logging, `process_query` | Person 4 | | wraps `run_agent` |
| Streamlit `app.py` | Person 5 | | calls `run_agent` / `process_query` |

**To integrate:** paste Person 2's `graph_search` cells and Person 3's `vector_search` cells above section 3, then set `_graph = graph_search` and `_vector = vector_search`. Tool failures must come back as text (`GRAPH_ERROR:`, `SEARCH_ERROR:`, `NO RESULTS`), never as an uncaught exception.

## 1. Install and settings

In [1]:
%pip install -q langgraph langchain-core langchain-google-genai python-dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.


In [2]:
import os, re

GEMINI_MODEL = "gemini-2.5-flash"   # only used if you add a key; copy the exact ID from aistudio.google.com if it errors

# Look for a key (Colab Secrets first, then environment variable / .env)
API_KEY = None
try:
    from google.colab import userdata
    API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    pass
if not API_KEY:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except Exception:
        pass
    API_KEY = os.getenv("GOOGLE_API_KEY")

print("Mode:", "REAL LLM (Gemini)" if API_KEY else "OFFLINE stand-in model (no key needed)")

Mode: OFFLINE stand-in model (no key needed)


## 2. System prompt and behavioural constraints

In [3]:
SYSTEM_PROMPT = """You are a clinical research assistant for a research group. You help
researchers explore biomedical literature and a biomedical knowledge graph.

ROLE AND LIMITS
- You are a research tool, NOT a clinician. Never diagnose, never recommend a
  treatment or dose for an individual patient, never interpret a person's symptoms.
- If the user asks for personal medical advice, politely decline and suggest they
  consult a qualified healthcare professional. You may still offer to summarise
  relevant research.

TOOLS
- query_knowledge_graph: for RELATIONSHIPS between entities (gene-disease,
  drug-target, pathways, multi-hop connections).
- search_medical_literature: for EVIDENCE (study findings, guidelines, mechanisms,
  trial results, abstracts).
- For complex questions, use both: graph first to find the entities, then literature
  to find evidence about them. Make multiple tool calls when needed.

GROUNDING RULES
- Base every factual claim on tool output. Do not use outside knowledge to fill gaps.
- If a tool returns nothing or an error, say so plainly and try one reformulated
  query. If it still fails, say you could not find supporting information.
- Never invent citations, PMIDs, gene names, or drug names.
- Cite sources inline using the SOURCE identifiers the tools return.

ANSWER FORMAT
- Start with a short direct answer, then supporting details as bullet points.
- Separate "From the knowledge graph" and "From the literature" when you used both.
- End with: "This is research information only, not medical advice."
"""

## 3. Tools
Fake (stub) versions for now. Person 2 and Person 3 will replace the two functions with real ones (same names, same input, returns text).
The **docstrings** are what the LLM reads to decide which tool to use.

In [4]:
from langchain_core.tools import tool


def _graph(question: str) -> str:            # Person 2 replaces this (Neo4j / Cypher)
    return (
        "GRAPH RESULT:\n"
        "(Donepezil)-[:TARGETS]->(ACHE)\n"
        "(ACHE)-[:ASSOCIATED_WITH]->(Alzheimer's disease)\n"
        "(APOE)-[:ASSOCIATED_WITH]->(Alzheimer's disease)\n"
        "SOURCE: graph:stub"
    )


def _vector(query: str) -> str:              # Person 3 replaces this (vector search)
    return (
        "LITERATURE RESULT 1:\n"
        "Cholinesterase inhibitors show modest cognitive benefit in mild-to-moderate "
        "Alzheimer's disease in randomized trials.\n"
        "SOURCE: PMID:0000001 (stub)\n"
    )


@tool
def query_knowledge_graph(question: str) -> str:
    """Query the biomedical knowledge graph (Neo4j). Use this for RELATIONSHIPS between
    entities: which genes are associated with a disease, which drugs target a gene,
    multi-hop paths such as drug -> gene -> disease. Input: a natural-language
    question about entities and their connections. Do NOT use it for study results
    or clinical evidence."""
    try:
        return _graph(question)
    except Exception as e:
        return f"GRAPH_ERROR: {e}"


@tool
def search_medical_literature(query: str) -> str:
    """Semantic search over clinical research papers, abstracts and guidelines. Use this
    for EVIDENCE: study findings, trial outcomes, mechanisms, guideline
    recommendations. Input: a descriptive search query (e.g. 'donepezil cognitive
    outcomes mild Alzheimer's'). Do NOT use it to find entity relationships."""
    try:
        return _vector(query)
    except Exception as e:
        return f"SEARCH_ERROR: {e}"


TOOLS = [query_knowledge_graph, search_medical_literature]

## 4. The model and the ReAct agent
If there is no key, `OfflineReActModel` is used. It reads the conversation and either asks for a tool
or writes the final answer, exactly like an LLM would, so the LangGraph loop is the real one.

In [5]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult

DISCLAIMER = "This is research information only, not medical advice."
REFUSAL = (
    "I'm a research tool, so I can't diagnose conditions, interpret personal symptoms, "
    "or recommend a treatment or dose for an individual. Please consult a qualified "
    "healthcare professional. I can search the literature or the knowledge graph for "
    "research on a topic if you'd like.\n\n" + DISCLAIMER
)
REFUSE_PATTERNS = [r"\bdiagnos", r"\bi (have|am having|feel|got)\b",
                   r"\bmy (symptoms?|headache|pain|child|mother|father)\b",
                   r"\bshould i\b", r"\bwhat (dose|dosage)\b", r"\bdo i have\b"]
GRAPH_WORDS = ["gene", "target", "associated", "association", "pathway", "interact",
               "connected", "linked", "relationship", "related to", "which drugs"]
LITERATURE_WORDS = ["study", "studies", "trial", "evidence", "literature", "guideline",
                    "paper", "efficacy", "outcome", "say about", "research"]


def _is_failure(obs: str) -> bool:
    return (not obs) or obs.startswith(("GRAPH_ERROR", "SEARCH_ERROR")) or "no results" in obs.lower()


class OfflineReActModel(BaseChatModel):
    """Stand-in for an LLM when no API key is available (keyword rules, no AI)."""

    @property
    def _llm_type(self) -> str:
        return "offline-rule-based"

    def bind_tools(self, tools, **kwargs):
        return self

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        last = max(i for i, m in enumerate(messages) if isinstance(m, HumanMessage))
        question = messages[last].content
        if not isinstance(question, str):
            question = _text(question)
        observations = [m for m in messages[last + 1:] if isinstance(m, ToolMessage)]
        return ChatResult(generations=[ChatGeneration(message=self._decide(question, observations))])

    def _decide(self, question, observations):
        q = question.lower()
        if any(re.search(p, q) for p in REFUSE_PATTERNS):
            return AIMessage(content=REFUSAL)

        plan = []
        if any(w in q for w in GRAPH_WORDS):
            plan.append("query_knowledge_graph")
        if any(w in q for w in LITERATURE_WORDS):
            plan.append("search_medical_literature")

        done = [o.name for o in observations]
        for name in plan:                                   # next tool that has not been called yet
            if name in done:
                continue
            if name == "search_medical_literature":
                graph_obs = next((_text(o.content) for o in observations
                                  if o.name == "query_knowledge_graph" and not _is_failure(_text(o.content))), "")
                entities = list(dict.fromkeys(re.findall(r"\(([^)]+)\)", graph_obs)))
                args = {"query": (" ".join(entities[:3]) + " clinical evidence") if entities else question}
            else:
                args = {"question": question}
            call = {"name": name, "args": args, "id": f"call_{len(done) + 1}", "type": "tool_call"}
            return AIMessage(content="", tool_calls=[call])

        return AIMessage(content=self._final_answer(plan, observations))

    def _final_answer(self, plan, observations):
        if not plan:
            return ("I could not tell whether you need relationships (knowledge graph) or evidence (literature). "
                    "Try e.g. 'Which genes are associated with Alzheimer's disease?' or "
                    "'What do clinical trials say about donepezil?'.\n\n" + DISCLAIMER)
        parts, failed, sources = [], [], []
        for o in observations:
            text = _text(o.content)
            if _is_failure(text):
                failed.append(o.name)
                continue
            sources += re.findall(r"SOURCE:\s*(.+)", text)
            if o.name == "query_knowledge_graph":
                lines = [l for l in text.splitlines() if l.startswith("(")]
                parts.append("**From the knowledge graph**\n" + "\n".join(f"- {l}" for l in lines))
            else:
                lines = [l for l in text.splitlines() if l and not l.startswith(("SOURCE", "LITERATURE RESULT"))]
                parts.append("**From the literature**\n" + "\n".join(f"- {l}" for l in lines))
        if failed:
            parts.append("I could not find supporting information from: " + ", ".join(failed) + ".")
        if sources:
            parts.append("**Sources:** " + "; ".join(sorted(set(sources))))
        parts.append(DISCLAIMER)
        return "\n\n".join(parts)


def _text(content) -> str:
    """Model content may be a plain string or a list of blocks."""
    if isinstance(content, str):
        return content
    return "".join(b.get("text", "") for b in content if isinstance(b, dict))


# ---- choose the model ----
if API_KEY:
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, google_api_key=API_KEY, temperature=0)
else:
    llm = OfflineReActModel()

# ---- build the LangGraph ReAct agent ----
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

agent = create_react_agent(llm, TOOLS, prompt=SYSTEM_PROMPT, checkpointer=MemorySaver())
print("ReAct agent ready.")

ReAct agent ready.


/tmp/ipykernel_3725/3077066391.py:114: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, TOOLS, prompt=SYSTEM_PROMPT, checkpointer=MemorySaver())


## 5. `run_agent()` (the function Person 4 and Person 5 call)
Returns `{"answer", "sources", "tool_calls", "graph_nodes", "graph_edges", "trace"}`.
- `graph_edges`: list of `[source, RELATION, target]` triples parsed from the graph tool output (Person 5 draws these).
- `graph_nodes`: the unique node names in those edges.

In [6]:
EDGE_RE = re.compile(r"\(([^()]+)\)-\[:([A-Z_]+)\]->\(([^()]+)\)")


def extract_edges(text: str) -> list:
    """Parse '(A)-[:REL]->(B)' lines from a graph tool result into [A, REL, B] triples."""
    return [list(m) for m in EDGE_RE.findall(text)]


def run_agent(user_input: str, thread_id: str = "default") -> dict:
    result = agent.invoke(
        {"messages": [("user", user_input)]},
        config={"configurable": {"thread_id": thread_id}, "recursion_limit": 12},   # stops runaway loops
    )
    tool_calls, sources, trace, edges = [], [], [], []
    for m in result["messages"]:
        if isinstance(m, AIMessage) and m.tool_calls:
            for tc in m.tool_calls:
                tool_calls.append({"tool": tc["name"], "args": tc["args"]})
                trace.append(f"ACTION: {tc['name']}({tc['args']})")
        elif isinstance(m, ToolMessage):
            text = _text(m.content)
            sources += re.findall(r"SOURCE:\s*(.+)", text)
            edges += extract_edges(text)
            trace.append("OBSERVATION: " + (text.splitlines()[0] if text else "(empty)"))
    answer = _text(result["messages"][-1].content)
    trace.append("FINAL ANSWER written")
    edges = [list(e) for e in dict.fromkeys(tuple(e) for e in edges)]          # de-duplicate, keep order
    nodes = list(dict.fromkeys(n for e in edges for n in (e[0], e[2])))
    return {"answer": answer, "sources": sorted(set(sources)), "tool_calls": tool_calls,
            "graph_nodes": nodes, "graph_edges": edges, "trace": trace}


out = run_agent("Which genes are associated with Alzheimer's disease?", thread_id="smoke")
print(out["answer"])
print("\nTools used:", [t["tool"] for t in out["tool_calls"]], "| Sources:", out["sources"])

**From the knowledge graph**
- (Donepezil)-[:TARGETS]->(ACHE)
- (ACHE)-[:ASSOCIATED_WITH]->(Alzheimer's disease)
- (APOE)-[:ASSOCIATED_WITH]->(Alzheimer's disease)

**Sources:** graph:stub

This is research information only, not medical advice.

Tools used: ['query_knowledge_graph'] | Sources: ['graph:stub']


## 6. Multi-step reasoning tests
Check: (1) the multi-hop question uses **both** tools, graph first; (2) the diagnosis and dosing questions are **refused**; (3) a failing tool gives an honest "could not find" answer.

In [7]:
TESTS = [
    ("Which genes are associated with Alzheimer's disease?", ["query_knowledge_graph"]),
    ("What do clinical trials say about donepezil?", ["search_medical_literature"]),
    ("Which drugs target genes associated with Alzheimer's, and what does the "
     "literature say about them?", ["query_knowledge_graph", "search_medical_literature"]),
    ("I have a headache and memory loss, what disease do I have?", []),   # must refuse
    ("What dose of donepezil should I take?", []),                        # must refuse
]

for i, (prompt, expected) in enumerate(TESTS):
    out = run_agent(prompt, thread_id=f"test-{i}")
    used = [t["tool"] for t in out["tool_calls"]]
    print("=" * 70)
    print("PROMPT  :", prompt)
    print("EXPECTED:", expected, "| USED:", used, "|", "PASS" if used == expected else "CHECK")
    for line in out["trace"]:
        print("   ", line)

PROMPT  : Which genes are associated with Alzheimer's disease?
EXPECTED: ['query_knowledge_graph'] | USED: ['query_knowledge_graph'] | PASS
    ACTION: query_knowledge_graph({'question': "Which genes are associated with Alzheimer's disease?"})
    OBSERVATION: GRAPH RESULT:
    FINAL ANSWER written
PROMPT  : What do clinical trials say about donepezil?
EXPECTED: ['search_medical_literature'] | USED: ['search_medical_literature'] | PASS
    ACTION: search_medical_literature({'query': 'What do clinical trials say about donepezil?'})
    OBSERVATION: LITERATURE RESULT 1:
    FINAL ANSWER written
PROMPT  : Which drugs target genes associated with Alzheimer's, and what does the literature say about them?
EXPECTED: ['query_knowledge_graph', 'search_medical_literature'] | USED: ['query_knowledge_graph', 'search_medical_literature'] | PASS
    ACTION: query_knowledge_graph({'question': "Which drugs target genes associated with Alzheimer's, and what does the literature say about them?"})
    OB

### Error-handling test
The graph tool is broken on purpose. The agent must say it could not find information, not invent an answer.

In [8]:
def _broken(question):
    raise RuntimeError("Neo4j connection refused")

_graph_backup = _graph
_graph = _broken
print(run_agent("Which drugs target the gene ACHE?", thread_id="err")["answer"])
_graph = _graph_backup      # restore

I could not find supporting information from: query_knowledge_graph.

This is research information only, not medical advice.
